# The Technical Agent (The "Chartist")
This notebook implements a Random Forest Regressor to identify chart patterns and momentum shifts. 

First, we will load the engineered dataset we created in the data labeler from the `csv-history` folder.

In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Load the labeled data from the history folder
df = pd.read_csv("../csv-history/technical_data_prepared.csv", index_col=0, parse_dates=True)
df.head()

,Open,High,Low,Close,adj close,Volume,Company,SMA_10,SMA_50,Daily_Return,Volatility,RSI,Target_5d_Return
Date,,,,,,,,,,,,,
1981-02-24,0.428571,0.428571,0.424107,0.424107,0.336037,4244800,AAPL,0.458705,0.530312,-0.035533,0.033942,31.325304,0.105263
1981-02-25,0.450893,0.453125,0.450893,0.450893,0.357260,4872000,AAPL,0.455134,0.529062,0.063158,0.038142,34.482759,0.029703
1981-02-26,0.457589,0.459821,0.457589,0.457589,0.362566,2710400,AAPL,0.453795,0.528482,0.014851,0.038252,36.666668,0.009756
1981-02-27,0.473214,0.477679,0.473214,0.473214,0.374946,3690400,AAPL,0.454464,0.528929,0.034146,0.038824,40.625003,-0.033019
1981-03-02,0.475446,0.477679,0.475446,0.475446,0.376715,2940000,AAPL,0.456473,0.529196,0.004717,0.037098,47.058827,-0.112676


Next, we separate our features from the target variable and split the data into training and testing sets. Because this is time-series data, we set `shuffle=False` to ensure we don't accidentally look into the future during training.

After splitting, we initialize and train the Random Forest model.

In [2]:
# Define Features (X) and Target (y)
features = ['Open', 'High', 'Low', 'Close', 'Volume', 'SMA_10', 'SMA_50', 'Volatility', 'RSI']
X = df[features]
y = df['Target_5d_Return']

# Split the data (No shuffling for time-series)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)

# Train the Random Forest Agent
rf_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
print("Training the Technical Agent...")
rf_model.fit(X_train, y_train)
print("Training complete!")

Training the Technical Agent...
Training complete!


Now we evaluate how well the model learned by predicting on our test set. 

Finally, we simulate generating a "Momentum Score" for today by feeding the model the latest unlabelled data from our live dataset.

In [3]:
# Evaluate the model
y_pred = rf_model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.6f}")
print(f"R-squared Score: {r2:.4f}")

# Test a prediction using the live data (The "Momentum Score")
live_df = pd.read_csv("../csv-history/technical_data_live.csv", index_col=0, parse_dates=True)
latest_features = live_df[features].iloc[-1:] # Grab the most recent day

momentum_score = rf_model.predict(latest_features)[0]

print(f"\nCalculated Momentum Score for today: {momentum_score:.4f}")
if momentum_score > 0:
    print("Agent Signal: Positive Expected Momentum")
else:
    print("Agent Signal: Negative Expected Momentum")

Mean Squared Error: 0.010027
R-squared Score: -2.2976

Calculated Momentum Score for today: 0.0037
Agent Signal: Positive Expected Momentum
